In [1]:
!pip install dotenv langchain langchain_groq langchain_community langgraph

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-tavily 0.2.0 requires langchain<0.4.0,>=0.3.20, but you have langchain 1.2.15 which is incompatible.
langchain-tavily 0.2.0 requires langchain-core<0.4.0,>=0.3.15, but you have langchain-core 1.8.1 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip



  Using cached langgraph_sdk-0.3.15-py3-none-any.whl.metadata (1.7 kB)
Using cached langgraph_sdk-0.3.15-py3-none-any.whl (98 kB)

  Attempting uninstall: langgraph-sdk

    Found existing installation: langgraph-sdk 0.4.14

   ---------------------------------------- 0/3 [langgraph-sdk]
    Uninstalling langgraph-sdk-0.4.14:
   ---------------------------------------- 0/3 [langgraph-sdk]
      Successfully uninstalled langgraph-sdk-0.4.14
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ---------------------------------------- 0/3 [langgraph-sdk]
   ----------------------------------------

In [3]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq


load_dotenv()


Groq_Api_Key = os.getenv("GROQ_API_KEY")
print("Api is getting laoded sucessfully")

Api is getting laoded sucessfully


In [5]:
Model = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.2
)


res = Model.invoke("Who is imran kahn?").content
print(res)

**Imran Khan** (born 5 October 1952) is a Pakistani politician, former cricketer, and philanthropist who served as the 22nd Prime Minister of Pakistan from August 2018 to April 2022.  

### Early Life & Cricket Career
- **Birthplace:** Lahore, Pakistan.  
- **Education:** Studied at Aitchison College, then at the University of Cambridge (St John’s College), where he earned a degree in Philosophy, Politics, and Economics.  
- **Cricket:**  
  - Debuted for Pakistan in 1971.  
  - Captained the national team from 1978 to 1992.  
  - Led Pakistan to its first Cricket World Cup victory in 1992.  
  - Known for his all‑round skills, charismatic leadership, and the “Imran Khan” brand of aggressive, high‑energy play.  
  - Retired from international cricket in 1992; later played in the Indian Cricket League and various T20 leagues.

### Political Career
- **Founding of PTI:** In 1996, he founded the Pakistan Tehreek‑e‑Insaf (PTI) party, positioning it as a movement against corruption and for 

In [6]:
!pip install pydantic


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
from pydantic import BaseModel,Field
from typing import Literal,Optional


class Complaint(BaseModel):
    product:str
    issue:str
    customer_request:str
    order_time: Optional[str]
    sentiment: str
    summary: str


In [11]:
Str_Model = Model.with_structured_output(schema=Complaint)

complaint = 'Mujhe kal parcel mila tha, lekin shoes ka size bohat chota hai.m refund nai replacement chahta hu.'
Str_Model.invoke(complaint)

Complaint(product='shoes', issue='Shoes size too small', customer_request='Mujhe kal parcel mila tha, lekin shoes ka size bohat chota hai.m refund nai replacement chahta hu.', order_time='yesterday', sentiment='negative', summary='Customer wants a replacement for shoes due to wrong size.')

In [ ]:
from typing import TypedDict
class RefundState(TypedDict):
    complaint:str
    complaint_analysis:Optional[dict]
    refund_assessment: Optional[dict]

In [15]:
from langchain_core.prompts import PromptTemplate

def complaint_node(state: RefundState):

    complaint = state["customer_complaint"]

    template = """
    You are a senior complaint analyzer.

    Analyze the following customer complaint:
    {complaint}

    Extract:
    - product
    - issue
    - customer_request
    - order_time
    - sentiment
    - summary

    Do not invent missing information.
    """

    prompt = PromptTemplate(
        template=template,
        input_variables=["complaint"]
    )

    formatted_prompt = prompt.format(complaint=complaint)

    res = Str_Model.invoke(formatted_prompt)

    return {
        "complaint_analysis": res.model_dump()
    }

In [18]:
from mock_data import refund_policy,order_details
class RefundAssesment(BaseModel):
    eligible:Literal[True,False]=Field(description="either the customer will be eligible(True) or not(False)")
    suggested_amount:float
    recommendation:str
    reason:str
    missing_information:Optional[str]=None

In [19]:
Assesment_Model = Model.with_structured_output(schema=RefundAssesment)

In [ ]:
def refund_assesment_node(state:RefundState):
    complaint_analysis = state['complaint_analysis']
    assessment_prompt = f"""
    You are a Refund Assessment Agent.

    Analyze the following information:

    CUSTOMER COMPLAINT:
    {complaint}

    ORDER DETAILS:
    {order_details}

    REFUND POLICY:
    {refund_policy}

    Your tasks:
    1. Check whether the customer is eligible for a refund.
    2. Check the refund policy carefully.
    3. Suggest the refund amount.
    4. Explain your decision.
    5. Identify missing information.

    Rules:
    - Never invent order information.
    - Follow the provided refund policy.
    - If required proof is missing, do not mark the refund as fully approved.
    - Only recommend; do not process the refund.
    """,
    prompt = PromptTemplate(
        template=assessment_prompt,
        input_variables=[complaint,order_details,refund_policy]
    )
    result = Assesment_Model.invoke(prompt)
    return {
        "refund_assessment": result.model_dump()
    }